In [1]:
import pandas as pd
import os

In [2]:
def read_file(file_name:str):
    df = (pd.read_csv(f"data/after-2016/{file_name}.csv", sep=";", encoding='latin-1')).drop(["id"],axis=1)
    
    return df

In [3]:
# Nota: confundi after com antes, logo esses dados na verdade refletem dados de 2016 para trás (2016-2007)
file_names = os.listdir("data/after-2016")

In [4]:
file_names[0]

'datatran2007.csv'

In [5]:
dfs = []
for i in file_names:
    file_name = i.split(".")[0]
    dfs.append(read_file(file_name))

C:\Users\RAFAEL\AppData\Local\Temp\ipykernel_14636\3963836253.py:2: DtypeWarning: Columns (0: br, 1: km) have mixed types. Specify dtype option on import or set low_memory=False.
  df = (pd.read_csv(f"data/after-2016/{file_name}.csv", sep=";", encoding='latin-1')).drop(["id"],axis=1)
C:\Users\RAFAEL\AppData\Local\Temp\ipykernel_14636\3963836253.py:2: DtypeWarning: Columns (0: br, 1: km) have mixed types. Specify dtype option on import or set low_memory=False.
  df = (pd.read_csv(f"data/after-2016/{file_name}.csv", sep=";", encoding='latin-1')).drop(["id"],axis=1)
C:\Users\RAFAEL\AppData\Local\Temp\ipykernel_14636\3963836253.py:2: DtypeWarning: Columns (0: br, 1: km) have mixed types. Specify dtype option on import or set low_memory=False.
  df = (pd.read_csv(f"data/after-2016/{file_name}.csv", sep=";", encoding='latin-1')).drop(["id"],axis=1)


In [6]:
df = (pd.concat(dfs, ignore_index=True))

In [7]:
def uf_to_regiao(uf):
    """
        Dado o UF passado nós dizemos a qual região ele pertence
    """

    if uf in ['GO', 'MT', 'MS', 'DF']:
        return "Centro-Oeste"
    elif uf in ['MA', 'CE', 'RN', 'PE', 'BA', 'AL', 'PI', 'SE', 'PB']:
        return "Nordeste"
    elif uf in ['PA', 'RO', 'AP', 'AC', 'RR', 'AM', 'TO']:
       return "Norte"
    elif uf in ['MG', 'ES', 'RJ', 'SP']:
        return "Sudeste"
    elif uf in ['PR', 'RS', 'SC']:
        return "Sul"

In [8]:
def resolve_km(x: str):
    x = x.split(',')[0].split('.')[0]
    return x

In [9]:
def get_month(date: str) -> str:

    split_slash = date.split('/')
    split_hyphen = date.split('-')

    # for split with '-'
    if len(split_hyphen) > 1:
        return split_hyphen[1]
    
    # for split with '/'
    return split_slash[1]

In [10]:
def get_days(date: str) -> str:

    split_slash = date.split('/')
    split_hyphen = date.split('-')

    # for split with '-'
    if len(split_hyphen) > 1:
        if len(split_hyphen[0]) == 4:
            return(split_hyphen[2])
        else:
            return(split_hyphen[0])
   
    # for split with '/'
    if len(split_slash[0]) == 4:
        return(split_slash[2])
    else:
        return(split_slash[0])

In [11]:
def fixing_year(year):

    if(year == '16'):
        return '20' + year
    return year

In [12]:
def get_hour(hours: str) -> str:
    return hours.split(':')[0]

In [13]:
def get_year_slash(date: str) -> str:
    
    split_slash = date.split('/')[-1]

    if(split_slash == '16'):
        return '20' + '16'
    return split_slash

In [14]:
def get_year_hif(date: str) -> str:
    
    split_slash = date.split('-')[0]

    if(split_slash == '16'):
        return '20' + '16'
    return split_slash

In [15]:
df['ano'] = df['data_inversa'].apply(get_year_hif)

In [16]:
# Pegando apenas alguns exemplos para ver se foi uma parte
df[['ano']].sample(10)

,ano
1554571,04/12/16
637921,15/12/2011
876670,2012
434811,18/01/2010
4911,15/01/2007
1547754,10/11/16
1018112,2013
1240232,2014
1233633,2014
712189,06/05/2011


In [17]:
df['ano'] = df['ano'].apply(get_year_slash)

In [18]:
# Pegando de um índice para comprovar que esse preencheu o restante do anterior
df[['ano']].iloc[679770]

ano    2011
Name: 679770, dtype: str

In [19]:
df['mes'] = df['data_inversa'].apply(get_month)

In [20]:
print(df[['mes']].min())
print(df[['mes']].max())

mes    01
dtype: str
mes    12
dtype: str


In [21]:
df.shape

(1562200, 26)

In [22]:
df.dropna(axis=0, how='any', inplace=True)

In [23]:
# Redução de apenas 5 linhas do total
df.shape

(1562195, 26)

In [24]:
# Problema nos dados, foram inseridos alguns '(null)' em algumas colunas
df[df['causa_acidente'] == '(null)']

,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,...,ano,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,mes
642934,27/12/2011,Terça,17:50:00,RJ,40,110.0,DUQUE DE CAXIAS,(null),Colisão lateral,Com Vítimas Feridas,...,2011,4,0,0,1,3,0,1,3,12
813603,2012-01-23,Segunda,09:00:00,RJ,356,160,SAO JOAO DA BARRA,(null),Atropelamento de animal,Sem Vítimas,...,2012,2,0,0,0,2,0,0,2,01


In [25]:
x = list(df.columns)

for i in x:
    df.drop(df[df[i] == '(null)'].index, inplace=True)

In [26]:
# Redução de 161 linhas do total, válido
df.shape

(1562034, 26)

In [27]:
# Como desejo criar uma série temporal na análise, preciso criar uma coluna que me facilite isso
df['ano_mes'] = pd.to_datetime(df['ano'] + '/' + df['mes'], format='mixed')

In [28]:
# Como ficou
df['ano_mes']

2         2007-08-01
3         2007-02-01
4         2007-11-01
5         2007-12-01
6         2007-03-01
             ...    
1562195   2016-11-01
1562196   2016-01-01
1562197   2016-12-01
1562198   2016-12-01
1562199   2016-12-01
Name: ano_mes, Length: 1562034, dtype: datetime64[us]

In [29]:
df['horas'] = df['horario'].apply(get_hour)

In [30]:
print(df['horas'].min())
print(df['horas'].max())

00
23


In [31]:
df['dia'] = df['data_inversa'].apply(get_days)

In [32]:
print(df['dia'].min())
print(df['dia'].max())

01
31


In [33]:
df['br'] = df['br'].astype(str)

In [34]:
df['km_int'] = df['km'].astype(str).apply(resolve_km)

In [35]:
df['km_int']

2           585
3            11
4            30
5            14
6           584
           ... 
1562195    1009
1562196     186
1562197      81
1562198     166
1562199      14
Name: km_int, Length: 1562034, dtype: str

In [36]:
df['regiao'] = df['uf'].apply(uf_to_regiao)

In [37]:
df.to_csv('data-treatment.csv', index=False)